# Doc2Query (2019)
---
[[paper]](https://arxiv.org/pdf/1904.08083)<br>
Doc2Query = Document to Query

Doc2Query — это подход, предложенный Google Research, который использует генеративную модель для расширения документов синтетическими запросами. Цель — улучшить эффективность информационного поиска за счет снижения проблемы "лексического разрыва" (vocabulary mismatch) между запросами пользователей и содержимым документов.

### Контекст

Одной из фундаментальных проблем в информационном поиске является **лексический разрыв** (vocabulary mismatch). Пользователь может искать информацию, используя формулировки, которые существенно отличаются от тех, что присутствуют в релевантных документах, даже если эти документы содержат необходимую информацию. Например, документ о "быстром прототипировании в стартапах" может быть релевантен запросу "как быстро запустить MVP", но традиционные методы поиска, основанные на точном совпадении слов (вроде BM25), могут его пропустить.

### Идея метода

Основная идея Doc2Query заключается в том, чтобы **генерировать множество потенциальных запросов для каждого документа**, которые пользователи могли бы задать, если бы искали этот документ. Эти синтетические запросы затем добавляются к оригинальному тексту документа, эффективно "расширяя" его. Таким образом, когда пользователь вводит реальный запрос, система поиска может найти документ, соответствующий одному из его синтетических запросов, даже если исходный документ не содержит точных формулировок. Это позволяет значительно улучшить охват (recall) и релевантность результатов, особенно для запросов, сформулированных неточно или с использованием синонимов.

### Постановка задачи

Doc2Query решает задачу **информационного поиска**, а именно улучшает качество ранжирования документов. Он выступает как метод **расширения документов** на этапе индексации, который затем может быть использован с любым стандартным движком поиска (например, BM25 для Sparse Retrieval или даже для улучшения Dense Retrieval).

### Альтернативные методы

На момент появления Doc2Query (2019) существовали следующие подходы к информационному поиску и решению проблемы лексического разрыва:

*   **Sparse Retrieval (например, TF-IDF, BM25):** Эти методы полагаются на подсчеты частоты слов и инвертированной частоты документов. Они очень эффективны для запросов с точным совпадением терминов, но крайне чувствительны к лексическому разрыву.
*   **Традиционное расширение запросов (Query Expansion):** Включает добавление синонимов, связанных терминов или результатов из тезаурусов к запросу пользователя. Может быть эффективным, но часто требует ручной настройки, основано на предопределенных словарях и может приводить к снижению точности из-за добавления нерелевантных терминов.
*   **Традиционное расширение документов (Document Expansion):** Подобно расширению запросов, но термины добавляются к документам (например, из анкоров ссылок, связанных страниц, метаданных). Имеет схожие ограничения с расширением запросов.
*   **Методы на основе Embeddings (например, Word2Vec (2013), DSSM (2013)):** Эти модели научились отображать слова или короткие тексты в плотные векторы (embeddings), что позволяло улавливать семантическую близость. Однако они не были нацелены на генерацию *запросов* для документов и обычно использовались для кодирования запроса и документа *по отдельности*, сравнивая их в векторном пространстве (как в **Two-Tower** архитектурах). Модели вроде OrQA (2019) и DrQA (2017) уже использовали более сложные нейросети для Question Answering, но Doc2Query уникален своей генеративной природой для создания *искусственных запросов*.

Новизна Doc2Query заключается в использовании **генеративной модели** для создания *синтетических запросов* из документов, вместо того чтобы полагаться на заранее определенные синонимы или просто кодировать существующий текст. Это позволяет автоматически создавать гораздо более разнообразные и контекстно-зависимые расширения, охватывая широкий спектр возможных пользовательских формулировок.

### Архитектура

Doc2Query использует **модель Sequence-to-Sequence (Seq2Seq)**. Как правило, это крупная предварительно обученная Transformer-модель, такая как T5 (2019) или другая encoder-decoder архитектура, способная генерировать текст.

*   **Вход:** Текст документа (или его фрагмента/пассажа).
*   **Выход:** Сгенерированный список потенциальных запросов, которые могли бы привести к этому документу.

Пример:
*   **Входной документ:** "Название "BERT" является акронимом от Bidirectional Encoder Representations from Transformers. Это модель трансформаторного типа, разработанная Google в 2018 году..."
*   **Сгенерированные запросы:** "что такое BERT", "когда был разработан BERT", "кто создал BERT", "акроним BERT", "BERT Google 2018"

### Алгоритм обучения

Обучение Doc2Query осуществляется в **контролируемом режиме (supervised learning)** на большом корпусе данных, где для каждого документа (или его фрагмента) известен **реальный пользовательский запрос**, который к нему привел.

1.  **Формирование обучающего набора данных:**
    *   Используются датасеты для Question Answering (например, Natural Questions, MS MARCO, SQuAD), где есть пары (вопрос, релевантный документ). В этих парах вопросы выступают в роли "запросов", а документы — в роли "документов".
    *   Каждая такая пара (документ `D`, запрос `Q`) рассматривается как обучающий пример, где модель должна научиться генерировать `Q`, получив `D` на вход.
2.  **Обучение Seq2Seq модели:**
    *   Модель обучается минимизировать функцию потерь (например, Negative Log-Likelihood) для задачи генерации последовательности.
    *   На вход подается документ `D`, и модель пытается сгенерировать последовательность токенов, соответствующую запросу `Q`.
    *   Для одного документа может существовать несколько релевантных запросов. В таком случае модель может быть обучена генерировать несколько запросов, либо для каждого запроса создается отдельный обучающий пример (документ, запрос).

### Алгоритм инференса (использования в поисковой системе)

Использование Doc2Query в реальной поисковой системе состоит из двух основных фаз: **индексация** и **поиск**.

1.  **Фаза индексации:**
    *   Для каждого документа `d` в корпусе документов:
        *   Документ `d` подается на вход обученной Doc2Query модели.
        *   Модель генерирует `N` синтетических запросов `q_1, q_2, ..., q_N`, релевантных этому документу.
        *   **Расширение документа:** Эти сгенерированные запросы `q_i` **конкатенируются** с оригинальным текстом документа `d`. Например, "Оригинальный текст документа. [SEP] q_1 [SEP] q_2 [SEP] ... [SEP] q_N".
        *   **Индексация:** Расширенный документ `d'` затем индексируется с помощью стандартного поискового движка (например, Lucene с BM25 или другая Dense Retrieval система).
2.  **Фаза поиска:**
    *   Когда пользователь вводит запрос `Q`:
        *   Запрос `Q` подается в поисковый движок, который ищет его в индексированном корпусе *расширенных* документов `d'`.
        *   Поскольку каждый `d'` содержит как оригинальный текст, так и множество синтетических запросов, вероятность совпадения `Q` с одним из них значительно возрастает, даже если `Q` не совпадает с оригинальным текстом.
        *   Поисковый движок возвращает ранжированный список документов.

### Результаты

Doc2Query демонстрирует значительные улучшения в задачах информационного поиска, особенно в случаях, когда лексический разрыв между запросами и документами велик.

*   На датасете MS MARCO (Microsoft Machine Reading Comprehension) Doc2Query улучшил метрику **MRR@10** (Mean Reciprocal Rank at 10) для системы BM25 на 10-20 пунктов по сравнению с базовой BM25 без расширения.
*   На других датасетах для Question Answering, таких как Natural Questions, использование Doc2Query для расширения документов перед индексацией также приводило к улучшению метрик Recall и F1 для Question Answering систем.
*   Преимущества были наиболее заметны для **Sparse Retrieval** систем, так как они наиболее чувствительны к проблеме лексического разрыва. Doc2Query эффективно "добавляет" словарный запас запросов в документ.
*   Может быть также использован для улучшения **Dense Retrieval** систем, например, путем обучения Dense Retrieval модели на парах (запрос, расширенный_документ) или путем генерации запросов, которые затем используются для создания **Hard Negatives** во время обучения.

## 📝 Критический анализ

```markdown
# Doc2Query (2019)
---
[[paper]](https://arxiv.org/pdf/1904.08083)<br>
Doc2Query = Document to Query

Doc2Query — подход от Google Research, использующий генеративную модель для расширения документов синтетическими запросами, чтобы улучшить информационный поиск, снижая проблему "лексического разрыва" между запросами и содержимым документов.

### Контекст

Проблема **лексического разрыва** возникает, когда пользовательские запросы формулируются иначе, чем текст в релевантных документах. Например, документ о "быстром прототипировании в стартапах" может быть релевантен запросу "как быстро запустить MVP", но традиционные методы поиска, такие как BM25, могут его пропустить.

### Идея

Doc2Query генерирует потенциальные запросы для каждого документа, которые добавляются к его тексту. Это позволяет системе поиска находить документ по синтетическим запросам, улучшая охват и релевантность результатов.

### Постановка задачи

Doc2Query улучшает качество ранжирования документов, выступая как метод **расширения документов** на этапе индексации, совместимый с любым стандартным движком поиска.

### Альтернативные методы

На момент появления Doc2Query (2019) существовали:
- **Sparse Retrieval (TF-IDF, BM25):** Эффективны для точного совпадения терминов, но чувствительны к лексическому разрыву.
- **Query Expansion:** Добавление синонимов, но требует ручной настройки и может снижать точность.
- **Document Expansion:** Добавление терминов к документам, имеет схожие ограничения.
- **Embeddings (Word2Vec (2013), DSSM (2013)):** Отображают слова в векторы, но не генерируют запросы.

Doc2Query уникален использованием **генеративной модели** для создания *синтетических запросов*, что позволяет автоматически создавать разнообразные и контекстно-зависимые расширения.

### Архитектура

Doc2Query использует **Seq2Seq** модель, например, T5 (2019).

* **Вход:** Текст документа.
* **Выход:** Список потенциальных запросов.

Пример:
* **Входной документ:** "Название "BERT" является акронимом от Bidirectional Encoder Representations from Transformers..."
* **Сгенерированные запросы:** "что такое BERT", "когда был разработан BERT", "кто создал BERT".

### Алгоритм обучения

Обучение в **контролируемом режиме** на корпусе данных, где для каждого документа известен **реальный пользовательский запрос**.

1. **Формирование обучающего набора:** Используются датасеты для Question Answering, где вопросы выступают в роли "запросов".
2. **Обучение Seq2Seq модели:** Модель обучается генерировать запросы, минимизируя функцию потерь.

### Алгоритм инференса

Использование Doc2Query в поисковой системе включает **индексацию** и **поиск**.

1. **Индексация:**
   * Для каждого документа генерируются синтетические запросы.
   * Запросы конкатенируются с текстом документа и индексируются.

2. **Поиск:**
   * Запрос пользователя ищется в индексированном корпусе расширенных документов.
   * Поисковый движок возвращает ранжированный список документов.

<img src="img/img.png" width=500>

### Результаты

Doc2Query улучшает задачи информационного поиска, особенно при большом лексическом разрыве.

* На MS MARCO Doc2Query улучшил **MRR@10** для BM25 на 10-20 пунктов.
* На Natural Questions улучшены метрики Recall и F1.
* Преимущества особенно заметны для **Sparse Retrieval** систем, но также могут улучшать **Dense Retrieval**.
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример использования Doc2Query для генерации синтетических запросов с использованием модели T5.

from transformers import T5ForConditionalGeneration, T5Tokenizer

# Загрузка предварительно обученной модели T5 и токенизатора
model_name = "t5-base"
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)

# Пример входного документа
document = "Название 'BERT' является акронимом от Bidirectional Encoder Representations from Transformers. Это модель трансформаторного типа, разработанная Google в 2018 году."

# Подготовка входных данных для модели
# Мы используем специальный префикс "generate queries:" для указания задачи генерации запросов
input_text = f"generate queries: {document}"
input_ids = tokenizer.encode(input_text, return_tensors="pt")

# Генерация синтетических запросов
# Параметр num_return_sequences определяет количество генерируемых запросов
outputs = model.generate(input_ids, max_length=50, num_return_sequences=5, num_beams=5)

# Декодирование и вывод сгенерированных запросов
generated_queries = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
print("Сгенерированные запросы:")
for i, query in enumerate(generated_queries, 1):
    print(f"{i}: {query}")

# Пример расширения документа с использованием сгенерированных запросов
# Конкатенируем оригинальный текст документа с сгенерированными запросами
expanded_document = document + " [SEP] " + " [SEP] ".join(generated_queries)
print("\nРасширенный документ:")
print(expanded_document)

# Этот расширенный документ затем может быть проиндексирован с помощью стандартного поискового движка,
# такого как Lucene с BM25, для улучшения поиска.
```

### Объяснение ключевых моментов:

1. **Использование модели T5**: Мы используем предварительно обученную модель T5, которая является мощной Seq2Seq моделью, способной генерировать текст. В данном случае она используется для генерации синтетических запросов на основе входного документа.

2. **Генерация запросов**: Мы подаем текст документа в модель с префиксом "generate queries:", чтобы указать задачу генерации запросов. Модель генерирует несколько потенциальных запросов, которые могли бы привести к этому документу.

3. **Расширение документа**: Сгенерированные запросы конкатенируются с оригинальным текстом документа. Это расширение позволяет улучшить поиск, так как увеличивает вероятность совпадения пользовательского запроса с текстом документа или с одним из синтетических запросов.

4. **Индексация и поиск**: Расширенный документ может быть проиндексирован с помощью стандартного поискового движка, что позволяет улучшить результаты поиска, особенно в условиях лексического разрыва.